# Question Answering on SQuAD v2.0 — Extractive vs RAG

**Author:** Naresh Gaur  
**Dataset:** Stanford Question Answering Dataset v2.0 (Rajpurkar et al., 2018)  
**Pipelines evaluated:**

1. **Extractive QA** — `deepset/roberta-base-squad2`, custom Top-K decoding with
   explicit `null-vs-best` no-answer flag.
2. **Retrieval-Augmented Generation (RAG)** — TF-IDF (and optionally dense MiniLM)
   retriever plus a `flan-t5-base` generator instructed to refuse when context
   is insufficient.

## Assignment coverage map

| Requirement                                  | Implementation                                  |
|----------------------------------------------|-------------------------------------------------|
| Pipeline A — Extractive QA, Top-K + scores   | `src/extractive_qa.py`                          |
| Pipeline A — Unanswerable handling           | Null-vs-best margin in `ExtractiveResult`       |
| Pipeline B — RAG (retrieve → inject → gen)   | `src/retriever.py`, `src/rag_qa.py`             |
| Pipeline B — Explicit refusal                | `NO_ANSWER_SENTINEL` prompt + post-check        |
| Recall@K, MRR, MAP **from scratch**          | `src/metrics.py` (+ unit tests in `tests/`)     |
| Recall@K plot (K = 1..10)                    | `outputs/figures/recall_at_k.png`               |
| LLM-as-Judge — Faithfulness & Relevance      | `src/llm_judge.py`                              |
| Reproducibility                              | `config.SEED = 42` propagated via `set_seed()`  |

All metrics are implemented from first principles (no `ranx` / `pytrec_eval`); the
from-scratch definitions are unit-tested against textbook examples in
`tests/test_smoke.py`.


## 1.  Setup

The cell below loads the pre-computed outputs of `run_pipeline.py`.  If those
outputs are not yet present (e.g. you're skimming the notebook before kicking
off the long run) we fall back to a synthetic illustration so the analysis
sections still render with consistent shapes.


In [ ]:
import json, os, sys, math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import config
from src.utils import normalize_answer, exact_match, f1_score
from src.metrics import (
    recall_at_k_curve, mean_reciprocal_rank, mean_average_precision,
    no_answer_classification,
)

OUT = ROOT / 'outputs'
FIG = OUT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
print('outputs dir:', OUT)
print('outputs present:', sorted(p.name for p in OUT.glob('*.json')))


In [ ]:
def _load_or_none(path: Path):
    if not path.exists():
        return None
    with open(path) as fh:
        return json.load(fh)

summary       = _load_or_none(OUT / 'summary.json')
ext_metrics   = _load_or_none(OUT / 'extractive_metrics.json')
rag_metrics   = _load_or_none(OUT / 'rag_metrics.json')
ext_preds     = _load_or_none(OUT / 'extractive_predictions.json')
rag_preds     = _load_or_none(OUT / 'rag_predictions.json')
judge_summary = _load_or_none(OUT / 'judge_summary.json')
judge_scores  = _load_or_none(OUT / 'judge_scores.json')
sample        = _load_or_none(OUT / 'sample.json')

have_real_run = ext_metrics is not None and rag_metrics is not None
print('have_real_run =', have_real_run)


## 2.  Pipeline A — Extractive QA

### Architecture

We use **`deepset/roberta-base-squad2`** — RoBERTa-base fine-tuned on SQuAD v2.
Our wrapper (`src/extractive_qa.py`) reimplements the standard span-enumeration
decoder rather than relying on `pipeline('question-answering')`, so the
no-answer logic is explicit and auditable:

1. Forward the (question, context) pair, obtaining `start_logits`, `end_logits`
   over every token position.
2. Enumerate all `(start, end)` pairs with `start ≤ end` and
   `end − start + 1 ≤ MAX_ANSWER_LEN` that lie inside the **context**
   sub-sequence (offsets check).
3. Score each candidate as `start_logit[start] + end_logit[end]`; soft-max
   over all candidates **plus a null span** at position 0.  The probability
   mass assigned to the null span is the model's calibrated confidence that
   the question is **unanswerable**.
4. De-duplicate candidate answer strings (different spans, identical text) and
   return the top-K with normalised scores.

The `no_answer` flag is set when the null probability exceeds the best
non-null probability by a configurable threshold (default 0).


In [ ]:
# show a couple of real predictions if available, else describe schema
if ext_preds:
    for p in ext_preds[:3]:
        print('Q :', p['question'][:90])
        print(f'   no_answer={p["no_answer"]}  null={p["null_score"]:.3f}  best={p["best_non_null_score"]:.3f}')
        for i, c in enumerate(p['candidates'][:3], 1):
            print(f'   {i}. "{c["text"]}"  score={c["score"]:.3f}')
        print()
else:
    print('No real predictions yet — run `python run_pipeline.py` first.')
    print('Schema of each prediction:')
    print('  qid, question, candidates[{text, score, start_char, end_char, start_logit, end_logit}],')
    print('  null_score, best_non_null_score, no_answer (bool), no_answer_margin')


## 3.  Pipeline B — Retrieval-Augmented Generation

### Architecture

The corpus is the set of **unique context paragraphs** across the sampled
questions (`build_corpus` in `src/data_loader.py`).  In SQuAD v2 each gold
answer lies inside exactly one paragraph, so this gives us a well-defined
per-question relevance label for retrieval evaluation.

Three stages:

1. **Retrieval.**  Default is **TF-IDF** (sparse cosine).  A dense MiniLM
   retriever is also available (`src/retriever.py`).  Both expose the same
   `rank(query, k)` API.
2. **Prompt construction.**  Top-K paragraphs are concatenated with
   `[Doc N] ...` headers into a prompt that *explicitly* permits a refusal
   token (`unanswerable`) when the context does not support an answer.
3. **Generation.**  `flan-t5-base` with beam search (4 beams,
   `max_new_tokens=64`).  The answer is post-checked for the refusal
   sentinel and mapped to the `no_answer` boolean.

Why this design handles SQuAD v2 unanswerable items: a generative model
*without* an explicit refusal channel cannot natively express "no answer".
By baking the refusal into the prompt and observing the surface form we
obtain a directly comparable `no_answer` signal to the extractive pipeline.


In [ ]:
if rag_preds:
    for r in rag_preds[:3]:
        print('Q :', r['question'][:90])
        print(f'   answer    : {r["answer"]!r}')
        print(f'   no_answer : {r["no_answer"]}')
        print(f'   top docs  : {r["retrieved_doc_ids"][:3]}')
        print()
else:
    print('No RAG predictions yet — run `python run_pipeline.py`.')


## 4.  Task 2A — Extractive Metrics

We treat the Top-K candidate spans returned by the extractive model as a
*ranked list of answers* and compute the standard IR triple:

- **Recall@K** — fraction of questions whose Top-K contains *any* gold
  answer (under SQuAD-style normalisation).
- **MRR** — mean of the reciprocal of the rank at which the *first*
  matching answer appears.
- **MAP** — mean of average precision across queries.

All three are implemented from scratch in `src/metrics.py` and unit-tested
against textbook examples.  Queries that are SQuAD v2 unanswerable are
*excluded* from the denominator: their relevance set is empty so any
well-defined ranking metric is undefined for them.


In [ ]:
def fmt_metrics_table(ext, rag_retr, rag_ans):
    rows = []
    if ext:
        rows.append({'Pipeline': 'Extractive (Top-K answers)',
                     'Recall@1': ext['recall_at_k']['1'] if '1' in ext['recall_at_k'] else ext['recall_at_k'][1],
                     'Recall@5': ext['recall_at_k']['5'] if '5' in ext['recall_at_k'] else ext['recall_at_k'][5],
                     'Recall@10': ext['recall_at_k']['10'] if '10' in ext['recall_at_k'] else ext['recall_at_k'][10],
                     'MRR': ext['mrr'], 'MAP': ext['map'],
                     'Top-1 EM': ext.get('top1_em'), 'Top-1 F1': ext.get('top1_f1')})
    if rag_retr:
        rows.append({'Pipeline': 'RAG retrieval (TF-IDF docs)',
                     'Recall@1': rag_retr['recall_at_k']['1'] if '1' in rag_retr['recall_at_k'] else rag_retr['recall_at_k'][1],
                     'Recall@5': rag_retr['recall_at_k']['5'] if '5' in rag_retr['recall_at_k'] else rag_retr['recall_at_k'][5],
                     'Recall@10': rag_retr['recall_at_k']['10'] if '10' in rag_retr['recall_at_k'] else rag_retr['recall_at_k'][10],
                     'MRR': rag_retr['mrr'], 'MAP': rag_retr['map'],
                     'Top-1 EM': rag_ans.get('em') if rag_ans else None,
                     'Top-1 F1': rag_ans.get('f1') if rag_ans else None})
    return pd.DataFrame(rows)

table = fmt_metrics_table(
    ext_metrics,
    rag_metrics['retrieval'] if rag_metrics else None,
    rag_metrics['answers']   if rag_metrics else None,
)
table.style.format(precision=3) if not table.empty else table


### 4.1  Recall@K curve

Recall@K shown for K = 1..10.  Two curves are overlaid where both pipelines
produce ranked lists of the same kind of object — extractive answers and RAG
retrieved documents — to highlight how the *unit* of retrieval differs even
when the curve shape looks similar.


In [ ]:
def _curve(d):
    # d['recall_at_k'] may be keyed by str or int depending on JSON round-trip
    items = d['recall_at_k']
    return [items[str(k)] if str(k) in items else items[k] for k in range(1, 11)]

fig, ax = plt.subplots(figsize=(8, 5))
if ext_metrics:
    ax.plot(range(1, 11), _curve(ext_metrics), marker='o', label='Extractive Top-K answers')
if rag_metrics:
    ax.plot(range(1, 11), _curve(rag_metrics['retrieval']), marker='s', label='RAG retrieval (TF-IDF docs)')
ax.set_xlabel('K'); ax.set_ylabel('Recall@K')
ax.set_title('Recall@K — Extractive vs RAG Retrieval')
ax.set_xticks(range(1, 11)); ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3); ax.legend(loc='lower right')
fig.tight_layout(); fig.savefig(FIG / 'recall_at_k_notebook.png', dpi=150)
plt.show()


### 4.2  Diminishing returns

We define the *diminishing-returns elbow* as the smallest `K*` such that
the marginal gain `Recall@(K*+1) − Recall@K*` falls below **2 percentage
points** for the first time.  This is the heuristic used in the open-domain
QA literature (Karpukhin et al., 2020) to choose retrieval `K` for downstream
readers.


In [ ]:
def find_elbow(curve, threshold=0.02):
    for k in range(1, len(curve)):
        gain = curve[k] - curve[k-1]
        if gain < threshold:
            return k, gain     # 1-indexed: gain from K=k to K=k+1 was small
    return len(curve), 0.0

for name, mdict in [('Extractive', ext_metrics),
                    ('RAG retrieval', rag_metrics['retrieval'] if rag_metrics else None)]:
    if mdict is None:
        continue
    c = _curve(mdict)
    elbow, gain = find_elbow(c)
    print(f'{name:18s} elbow at K={elbow}  (marginal gain there = {gain:+.3f})')
    print(f'                 Recall@{elbow}={c[elbow-1]:.3f}  Recall@10={c[-1]:.3f}'
          f'  -> {c[-1]-c[elbow-1]:+.3f} headroom beyond elbow')


### 4.3  When Recall@1 is low but MRR is high

**Intuition.**  `MRR` averages reciprocal ranks; `Recall@1` is binary (hit or
miss at the top).  A system whose correct answer often sits at *rank 2 or 3*
rather than rank 1 will have `Recall@1` close to its `MRR`-implied position
but higher `Recall@K` for `K > 1`.  Concretely, if a system answers correctly
at rank 2 for half its queries and rank 1 for the other half:

```
Recall@1 = 0.5     MRR = (0.5·1 + 0.5·0.5) = 0.75
```

i.e. **MRR can be ~50% higher than Recall@1** even though the system only
answers correctly half the time at the top.  This signals a *re-ranking
problem* rather than a *retrieval* problem — the right answer is in the
list, it's just not first.  Re-ranking heads (e.g. a small cross-encoder
over the top-10) typically close most of this gap.

The cell below identifies extractive questions where the gold answer was
found in the Top-K but **not** at rank 1.


In [ ]:
def gold_rank(ext_pred, golds):
    g_norm = {normalize_answer(g) for g in golds if g.strip()}
    if not g_norm:
        return None
    for i, c in enumerate(ext_pred['candidates'], 1):
        if normalize_answer(c['text']) in g_norm:
            return i
    return None

rerank_cases = []
if ext_preds and sample:
    sample_by_qid = {ex['qid']: ex for ex in sample}
    for p in ext_preds:
        ex = sample_by_qid.get(p['qid'])
        if not ex or ex['is_unanswerable']:
            continue
        r = gold_rank(p, ex['answers'])
        if r is not None and r > 1:
            rerank_cases.append((r, ex['question'][:80], ex['answers'], [c['text'] for c in p['candidates'][:3]]))
    rerank_cases.sort()
    print(f'{len(rerank_cases)} answerable questions where the gold answer is in Top-K but NOT at rank 1.')
    for r, q, golds, top3 in rerank_cases[:5]:
        print(f'  rank={r}  Q: {q}')
        print(f'         gold = {golds}')
        print(f'         top3 = {top3}')
else:
    print('Run the pipeline to populate this analysis.')


## 5.  Task 2B — Generative QA Evaluation

Two qualities are scored on the RAG outputs:

1. **Faithfulness (Groundedness).**  An LLM judge (the same `flan-t5-base`,
   used in self-judge mode) is prompted to grade on a 1-5 scale how well
   every claim in the answer is supported by the retrieved context.
2. **Answer Relevance.**  The same judge is prompted to grade on a 1-5
   scale how directly the answer addresses the question.  This penalises
   irrelevant, redundant or hallucinated content even when the surface form
   looks plausible.

**Caveat — self-judging bias.**  Using the generator as its own judge is
biased upward (Zheng et al., 2023, *Judging LLM-as-a-Judge*): the model
tends to forgive its own outputs because they share lexical patterns.  We
therefore additionally report deterministic *heuristic* baselines —
token-overlap of the answer with the context (faithfulness proxy) and
with the question (relevance proxy) — and discuss the two side by side.


In [ ]:
if judge_summary:
    table = pd.DataFrame([
        {'Metric': 'Faithfulness (LLM judge, 1-5)',
         'Value' : judge_summary['faithfulness_llm_mean']},
        {'Metric': 'Faithfulness (token-overlap heuristic, 0-1)',
         'Value' : judge_summary['faithfulness_heuristic_mean']},
        {'Metric': 'Relevance (LLM judge, 1-5)',
         'Value' : judge_summary['relevance_llm_mean']},
        {'Metric': 'Relevance (token-overlap heuristic, 0-1)',
         'Value' : judge_summary['relevance_heuristic_mean']},
        {'Metric': 'Judged answers (n)',
         'Value' : judge_summary['n_judged']},
    ])
    display(table.style.format({'Value': '{:.3f}'}))
else:
    print('No judge scores yet — run with --judge-subset N (see run_pipeline.py).')


In [ ]:
# Distribution of LLM-judge scores
if judge_scores:
    f_scores = [s['faithfulness_llm'] for s in judge_scores if s['faithfulness_llm'] is not None]
    r_scores = [s['relevance_llm']    for s in judge_scores if s['relevance_llm']    is not None]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].hist(f_scores, bins=range(1, 7), align='left', color='steelblue', edgecolor='black')
    axes[0].set_title('Faithfulness (1-5)'); axes[0].set_xlabel('score'); axes[0].set_ylabel('count')
    axes[1].hist(r_scores, bins=range(1, 7), align='left', color='seagreen',  edgecolor='black')
    axes[1].set_title('Relevance (1-5)');    axes[1].set_xlabel('score')
    for ax in axes:
        ax.set_xticks(range(1, 6)); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(FIG / 'judge_distribution.png', dpi=150); plt.show()
else:
    print('Run the LLM judge step to populate this plot.')


## 6.  Extractive vs Generative — head-to-head

We compare the two pipelines along three axes:

1. **Top-1 quality** — EM / F1 of the single best answer.
2. **Unanswerable handling** — precision / recall of the `no_answer` flag
   on the SQuAD v2 unanswerable subset.
3. **Failure mode** — where does each system *go wrong*?

The third axis is the most informative one and is analysed below.


In [ ]:
def collect_top1_table():
    rows = []
    if ext_metrics:
        rows.append({
            'Pipeline': 'Extractive (RoBERTa-squad2)',
            'Top-1 EM': ext_metrics.get('top1_em'),
            'Top-1 F1': ext_metrics.get('top1_f1'),
            'NoAns Precision': ext_metrics.get('no_answer_precision'),
            'NoAns Recall'   : ext_metrics.get('no_answer_recall'),
            'NoAns F1'       : ext_metrics.get('no_answer_f1'),
        })
    if rag_metrics:
        a = rag_metrics['answers']
        rows.append({
            'Pipeline': 'RAG (TF-IDF + flan-t5-base)',
            'Top-1 EM': a.get('em'),
            'Top-1 F1': a.get('f1'),
            'NoAns Precision': a.get('no_answer_precision'),
            'NoAns Recall'   : a.get('no_answer_recall'),
            'NoAns F1'       : a.get('no_answer_f1'),
        })
    return pd.DataFrame(rows)

df_top1 = collect_top1_table()
df_top1.style.format(precision=3) if not df_top1.empty else df_top1


### 6.1  Hallucination vs extractive error

We classify the failure cases on the **answerable** subset into four
categories so the two pipelines can be compared on the same axis:

| Failure mode               | Definition                                                                  |
|----------------------------|-----------------------------------------------------------------------------|
| `wrong_span`               | (extractive) prediction is a verbatim span of the context but not the gold  |
| `false_no_answer`          | model said "unanswerable" when a gold answer exists                         |
| `hallucination`            | (RAG) answer contains tokens that **do not appear** in the retrieved docs   |
| `paraphrase` (false miss)  | answer is semantically right but the normalised string doesn't match gold   |

`hallucination` is impossible for the extractive pipeline by construction —
the output is *always* a substring of the context.  That asymmetry is the
single biggest qualitative difference between the two approaches.


In [ ]:
def categorise(ext_pred, rag_pred, ex, ctx_lookup):
    """Return (extractive_failure_mode, rag_failure_mode) or (None, None) when correct."""
    golds = [normalize_answer(g) for g in ex['answers'] if g.strip()]
    ext_top = ext_pred['candidates'][0]['text'] if (ext_pred['candidates'] and not ext_pred['no_answer']) else ''
    ext_norm = normalize_answer(ext_top)
    ctx = ex['context']
    ext_mode = None
    if ext_pred['no_answer']:
        ext_mode = 'false_no_answer' if golds else None
    elif golds and ext_norm not in golds:
        ext_mode = 'wrong_span' if ext_top.strip() and ext_top in ctx else 'paraphrase'
    elif not golds and not ext_pred['no_answer']:
        ext_mode = 'false_answer_on_unans'
    
    rag_mode = None
    rag_ans = rag_pred['answer']
    rag_norm = normalize_answer(rag_ans)
    retrieved_ctx = ' '.join(ctx_lookup.get(d, '') for d in rag_pred['retrieved_doc_ids'])
    retrieved_tokens = set(normalize_answer(retrieved_ctx).split())
    answer_tokens = set(rag_norm.split())
    if rag_pred['no_answer']:
        rag_mode = 'false_no_answer' if golds else None
    elif golds and rag_norm not in golds:
        novel = answer_tokens - retrieved_tokens
        if novel and len(novel) / max(1, len(answer_tokens)) > 0.5:
            rag_mode = 'hallucination'
        else:
            rag_mode = 'paraphrase'
    elif not golds and not rag_pred['no_answer']:
        rag_mode = 'false_answer_on_unans'
    return ext_mode, rag_mode

if ext_preds and rag_preds and sample:
    ctx_lookup = {}
    if sample is not None:
        for ex in sample:
            ctx_lookup.setdefault(ex['context'], ex['context'])
    # actually we need doc_id -> text; the side-by-side csv has top1 doc id but not text. Use context per qid:
    by_qid = {ex['qid']: ex for ex in sample}
    ext_modes, rag_modes = [], []
    for ep, rp in zip(ext_preds, rag_preds):
        ex = by_qid.get(ep['qid'])
        if not ex: continue
        # crude ctx_lookup: doc_id -> the question's own context if it appears in retrieved set
        local_lookup = {d: ex['context'] for d in rp['retrieved_doc_ids']}
        em, rm = categorise(ep, rp, ex, local_lookup)
        ext_modes.append(em); rag_modes.append(rm)
    ext_dist = Counter(m for m in ext_modes if m is not None)
    rag_dist = Counter(m for m in rag_modes if m is not None)
    pd.DataFrame({'extractive': ext_dist, 'rag': rag_dist}).fillna(0).astype(int)
else:
    print('Need both pipelines run + sample.json present.')


## 7.  Discussion

Detailed write-up is in `report/REPORT.md`.  In summary:

- **Extractive QA** is the stronger Top-1 system on SQuAD v2 in absolute terms
  *and* the safer one: by construction it cannot hallucinate.  Its dominant
  failure mode is choosing the *wrong* span — a paraphrase that is locally
  plausible.
- **RAG** wins on questions that require lightly rephrasing the supporting
  text and on questions where the gold answer is spread across multiple
  sentences.  It loses on extractive-style factoid questions and exposes a
  new failure mode (hallucination) absent in the extractive baseline.
- **Unanswerable handling** is generally tighter on the extractive side
  because the no-answer signal is a calibrated probability; the RAG side
  relies on the model deciding to emit a sentinel token, which is brittle
  to retrieval noise.
- **Suggested improvements:**
  1. A cross-encoder re-ranker over the Top-10 extractive candidates to
     close the MRR-vs-Recall@1 gap.
  2. Dense (MiniLM/SimCSE) retrieval to lift RAG Recall@1.
  3. A second LLM-as-Judge pass using a *different* model (e.g. Llama-3-8B)
     to break the self-judge bias.
  4. Constrained decoding (force the answer to be a substring of the
     retrieved context) to eliminate hallucination in the RAG pipeline.
